In [18]:
import geopandas as gpd
import pystac_client


from helpers.simplecube import load_xarray, save_xarray, simple_cube

STAC_URL = "https://data.inpe.br/bdc/stac/v1"

def format_bbox(bbox_string):
    return ",".join(map(str, bbox))

def get_composition(cube_id, reject: any = []):
    service = pystac_client.Client.open(STAC_URL)
    collection = service.get_collection(cube_id)
    item_assets = collection.to_dict()['item_assets']
    composition = [ id for id in list(item_assets.keys()) if id not in reject]
    return composition

In [29]:
tile = gpd.read_file("./dataset/tile_C51L41_WGS84.gpkg")
if tile.crs != "EPSG:4326":
    tile = tile.to_crs(epsg=4326)
bbox = list([ float(b) for b in tile.total_bounds ])
bbox

[-61.25000000000001, -8.25, -61.0, -7.999999999999998]

In [30]:
cube = "S2-16D-2"
year = 2024

start, end = (f"{year}-01-01", f"{year}-12-31")
composition = get_composition(cube, reject = ["thumbnail", "CMASK", "CLEAROB", "TOTALOB", "DATASOURCE", "PROVENANCE", "SCL"])
bbox = format_bbox(bbox)
composition
bbox

'-61.25000000000001,-8.25,-61.0,-7.999999999999998'

In [ ]:
cube = simple_cube(
     stac_url   = STAC_URL,
     collection = cube,
     start_date = start,
     end_date   = end,
     bbox       = bbox,
     bands      = composition
)
save_xarray(cube, f'./dataset/rasters/cube_sentinel2_{year}.nc')
cube = load_xarray(f'./dataset/rasters/cube_sentinel2_{year}.nc')

r, g, b = ['red', 'green', 'blue']

rgb_composition = xr.concat([cube[var].isel(time=5, band=0) for var in [r, g, b]], dim="band")
rgb_composition = rgb_composition.assign_coords(band=["R", "G", "B"])
rgb_composition = rgb_composition.transpose("y", "x", "band")
rgb_composition = rgb_composition / rgb_composition.max() # normalize
rgb_composition.plot.imshow()
plt.gca().set_aspect('equal')

Fetching... : 100%|██████████| 23/23 [00:00<00:00, 220.69 scenes/s]
